# Lekcja 04 - Wzorzec projektowy użycia narzędzi

W tej lekcji nauczysz się wzorca projektowego **Użycia narzędzi** dla agentów AI korzystających z Microsoft Agent Framework (Python). Omówimy:

- Definiowanie funkcji narzędziowych z dekoratorem `@tool` i parametrami z typami
- Dostarczanie schematów narzędzi, aby model wiedział, do czego służy każde narzędzie
- Kontrolę wykonania narzędzi za pomocą `approval_mode`
- Zwracanie **ustrukturyzowanych wyników** za pomocą modeli Pydantic oraz `response_format`

Scenariusz to **agent rezerwacji podróży**, który może wyszukiwać destynacje, sprawdzać dostępność oraz pobierać informacje o lotach.


## Konfiguracja


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Definiowanie narzędzi za pomocą dekoratora @tool

Dekorator `@tool` zamienia zwykłą funkcję Pythona w narzędzie, które agent może wywołać.  
Najważniejsze punkty:

- **Docstring** staje się opisem narzędzia, które model widzi.  
- **Adnotacje typów** (w tym `Annotated` z opisami) definiują schemat narzędzia.  
- `approval_mode` kontroluje, czy użytkownik musi zatwierdzić każde wywołanie przed jego wykonaniem.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Tworzenie agenta z wieloma narzędziami

Przekaż wszystkie trzy narzędzia do klienta, aby model mógł wywołać te, które są potrzebne do odpowiedzi na pytanie użytkownika.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Strukturalny wynik z narzędziami

Ustawiając `response_format` na model Pydantic, agent jest zmuszony do zwrócenia dobrze typowanego obiektu JSON zamiast tekstu w formie wolnej. Jest to przydatne, gdy kod wykonujący się poniżej musi programowo wykorzystać wynik.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Wzorce zatwierdzania narzędzi

Parametr `approval_mode` w `@tool` kontroluje, czy wywołania narzędzi wymagają zatwierdzenia przez człowieka przed wykonaniem:

| Tryb | Zachowanie |
|---|---|
| `"never_require"` | Narzędzie działa automatycznie — nie wymaga potwierdzenia od użytkownika. |
| `"always_require"` | Każde wywołanie musi być zatwierdzone przez użytkownika przed wykonaniem. |

Używaj `"always_require"` dla narzędzi, które mają skutki uboczne (np. rezerwacja lotu, obciążenie karty kredytowej), aby człowiek pozostał w pętli.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Podsumowanie

W tej lekcji nauczyłeś się, jak:

1. **Definiować narzędzia** za pomocą dekoratora `@tool` z typowanymi parametrami i docstringami, które służą jako schemat narzędzia.
2. **Łączyć wiele narzędzi**, aby agent mógł wywoływać je kolejno w celu odpowiedzi na złożone zapytania.
3. **Zwracać ustrukturyzowany wynik**, przekazując model Pydantic jako `response_format`.
4. **Kontrolować zatwierdzanie narzędzi** za pomocą `approval_mode`, aby utrzymać udział człowieka wrażliwych operacjach.

Te wzorce stanowią podstawę do budowy niezawodnych, gotowych do produkcji agentów, którzy mogą bezpiecznie współdziałać z systemami zewnętrznymi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:  
Ten dokument został przetłumaczony za pomocą automatycznej usługi tłumaczeniowej [Co-op Translator](https://github.com/Azure/co-op-translator). Mimo że dążymy do dokładności, prosimy mieć na uwadze, że tłumaczenia automatyczne mogą zawierać błędy lub niedokładności. Oryginalny dokument w języku źródłowym powinien być uznawany za autorytatywne źródło. W przypadku informacji o kluczowym znaczeniu zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za ewentualne nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
